In [1]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
print(f"Project root: {PROJECT_ROOT}")
DATA_DIR = PROJECT_ROOT / "data"

def read_json(file_path: Path | str) -> dict | list:
    data = None
    with open(file_path, "r") as f:
        data = json.load(f)
    return data

def read_jsonl(file_path: Path | str) -> list:
    res = []
    with open(file_path, "r") as f:
        for line in f:
            res.append(json.loads(line))
    return res

def write_json(data: dict | list, file_path: Path | str) -> None:
    with open(file_path, "w") as f:
        json.dump(data, f, indent=4)
        print(f"wrote to {file_path}")
        

Project root: /home/v-homatthew/ctx_editor


In [2]:
original_lic_data = read_json(DATA_DIR / "sharded_instructions_600.json")

In [3]:
code_only = [item for item in original_lic_data if item.get("task") == "code"]

In [4]:
# add full spec QA

from datasets import load_dataset

task_subset = ["math", "code", "actions", "database"]
he_dataset = load_dataset("openai/openai_humaneval")

def add_full_spec_qa(data: list) -> None:
    for item in data:
        task = item["task"]
        if task not in task_subset:
            continue

        if task == "math":
            item["full_spec_q"] = item["question"]
            item["ground_truth_a"] = item["answer"]
        elif task == "code":
            # task id options:
            # - sharded-HumanEval/{number}
            # - sharded-livecodebench/{number}
            if item["task_id"].startswith("sharded-HumanEval/"):
                item["full_spec_q"] = item.get("prompt", None)
                number = int(item["task_id"].split("/")[1])
                item["ground_truth_a"] = he_dataset["test"][number]["canonical_solution"]
            else:
                item["full_spec_q"] = item.get("question_content", None)
                # NOTE [2026.01.27]
                # - we can use execution-v2 dataset to get ground truth answers
                # - there's not a unique answer per question, the execution-v2 dataset contains multiple per question_id
                # - we can just pick any one of them (e.g. first or last)
                # - deferring for now
                item["ground_truth_a"] = None
                if item["full_spec_q"] is None:
                    print(
                        f"livecodebench task id: {item['task_id']}, no full spec question available"
                    )
        elif task == "actions":
            item["full_spec_q"] = item["fully_specified_question"][0][0]["content"]
            item["ground_truth_a"] = item["reference_answer"]
        elif task == "database":
            item["full_spec_q"] = item["fully_specified_question"]
            item["ground_truth_a"] = item["reference_sql"]
        else:
            print(f"skipping {item['task_id']}")
            item["full_spec_q"] = None
            item["ground_truth_a"] = None

In [9]:
add_full_spec_qa(code_only)

In [ ]:
code_only[0]

{'task_id': 'sharded-HumanEval/105',
 'prompt': '\ndef by_length(arr):\n    """\n    Given an array of integers, sort the integers that are between 1 and 9 inclusive,\n    reverse the resulting array, and then replace each digit by its corresponding name from\n    "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine".\n\n    For example:\n      arr = [2, 1, 1, 4, 5, 8, 2, 3]   \n            -> sort arr -> [1, 1, 2, 2, 3, 4, 5, 8] \n            -> reverse arr -> [8, 5, 4, 3, 2, 2, 1, 1]\n      return ["Eight", "Five", "Four", "Three", "Two", "Two", "One", "One"]\n    \n      If the array is empty, return an empty array:\n      arr = []\n      return []\n    \n      If the array has any strange number ignore it:\n      arr = [1, -1 , 55] \n            -> sort arr -> [-1, 1, 55]\n            -> reverse arr -> [55, 1, -1]\n      return = [\'One\']\n    """\n',
 'test': 'def check(candidate):\n\n    # Check some simple cases\n    assert True, "This prints if this assert fai

In [11]:
write_json(code_only, DATA_DIR / "full_code_subset.json")

wrote to /home/v-homatthew/ctx_editor/data/full_code_subset.json


In [5]:
# repeat for actions and database
actions_only = [item for item in original_lic_data if item.get("task") == "actions"]
add_full_spec_qa(actions_only)
write_json(actions_only, DATA_DIR / "full_actions_subset.json")
database_only = [item for item in original_lic_data if item.get("task") == "database"]
add_full_spec_qa(database_only)
write_json(database_only, DATA_DIR / "full_database_subset.json")

wrote to /home/v-homatthew/ctx_editor/data/full_actions_subset.json
wrote to /home/v-homatthew/ctx_editor/data/full_database_subset.json
